# VisionAssist Phase 8 — Colab-resilient QLoRA infrastructure

This notebook runs the Phase 8 sequence:

1. clone/pull the repository;
2. restore the prepared-data archive from Drive;
3. install the training environment;
4. inspect the GPU;
5. configure persistent Drive checkpoints;
6. run one-batch validation;
7. run the 32-example overfit experiment;
8. interrupt/restart safely and resume from the newest checkpoint;
9. retain only the newest two checkpoints plus the best checkpoint.

Select **Runtime → Change runtime type → GPU** before running.

In [ ]:
#@title 1. Settings
from pathlib import Path

REPO_URL = "https://github.com/<YOUR_USERNAME>/visionassist-industrial-visual-inspection.git"
REPO_BRANCH = "main"

DRIVE_ROOT = Path("/content/drive/MyDrive/visionassist")
DRIVE_DATA_ARCHIVE = DRIVE_ROOT / "data/visionassist_prepared_data.tar.gz"
PROJECT_ROOT = Path("/content/visionassist-industrial-visual-inspection")

OVERFIT_CONFIG = PROJECT_ROOT / "configs/training/qwen25vl3b_qlora_overfit.yaml"
OVERFIT_RUN_ID = "qwen25vl3b_qlora_overfit_v1"
DRIVE_CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints" / OVERFIT_RUN_ID

assert "<YOUR_USERNAME>" not in REPO_URL, "Set REPO_URL first."

In [ ]:
#@title 2. Mount Drive
from google.colab import drive
drive.mount("/content/drive")
for path in [DRIVE_ROOT/"data", DRIVE_ROOT/"checkpoints", DRIVE_ROOT/"outputs"]:
    path.mkdir(parents=True, exist_ok=True)
assert DRIVE_DATA_ARCHIVE.is_file(), f"Missing: {DRIVE_DATA_ARCHIVE}"

In [ ]:
#@title 3. Verify GPU
import shutil, subprocess, torch
assert torch.cuda.is_available(), "Choose a GPU runtime."
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(props.total_memory/1024**3, 2))
print("BF16:", torch.cuda.is_bf16_supported())
print("Free disk GiB:", round(shutil.disk_usage('/content').free/1024**3, 2))
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
#@title 4. Install uv and clone/pull repository
import subprocess, sys, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"], check=True)
if (PROJECT_ROOT/".git").is_dir():
    subprocess.run(["git","fetch","origin",REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git","checkout",REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git","pull","--ff-only","origin",REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
else:
    if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git","clone","--branch",REPO_BRANCH,REPO_URL,str(PROJECT_ROOT)], check=True)
print(subprocess.check_output(["git","rev-parse","HEAD"], cwd=PROJECT_ROOT, text=True).strip())

In [ ]:
#@title 5. Install dependencies
import os, subprocess
os.chdir(PROJECT_ROOT)
subprocess.run(["uv","sync","--extra","training","--extra","dev"], check=True)

In [ ]:
#@title 6. Restore prepared data to fast local storage
import tarfile
required = [
    PROJECT_ROOT/"data/raw/visa",
    PROJECT_ROOT/"data/processed/visa_instructions/train.jsonl",
    PROJECT_ROOT/"data/processed/visa_instructions/validation.jsonl",
]
if not all(path.exists() for path in required):
    with tarfile.open(DRIVE_DATA_ARCHIVE, "r:gz") as archive:
        archive.extractall(PROJECT_ROOT)
assert all(path.exists() for path in required)
print("Prepared data ready.")

In [ ]:
#@title 7. Configure persistent bounded checkpoints
import yaml
cfg = yaml.safe_load(OVERFIT_CONFIG.read_text(encoding="utf-8"))
cfg["checkpoints"]["resume"] = "latest"
cfg["checkpoints"]["keep_latest"] = 2
cfg["checkpoints"]["keep_best"] = 1
cfg["checkpoints"]["persistent_output_dir"] = str(DRIVE_CHECKPOINT_ROOT)
cfg["checkpoints"]["sync_every_save"] = True
cfg["training"]["save_total_limit"] = 3
OVERFIT_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
#@title 8. Inspect Phase 8 environment
import subprocess
subprocess.run(["uv","run","visionassist","training-environment","--config",str(OVERFIT_CONFIG)], cwd=PROJECT_ROOT, check=True)

In [ ]:
#@title 9. Run local unit tests before GPU use
import subprocess
subprocess.run(["uv","run","pytest","tests/test_phase8_training.py"], cwd=PROJECT_ROOT, check=True)

In [ ]:
#@title 10. One-batch forward-pass smoke test
import subprocess
subprocess.run(["uv","run","visionassist","training-smoke-test","--config",str(OVERFIT_CONFIG)], cwd=PROJECT_ROOT, check=True)

## 32-example overfit run

This run saves every 10 steps. Local `/content` retains at most 3 checkpoints.
Drive retains the newest two plus the best checkpoint. Run the same command after
a disconnect; `--resume latest` restores the newest persistent checkpoint.

In [ ]:
#@title 11. Train or resume the overfit run
import subprocess
subprocess.run([
    "uv","run","visionassist","train-qlora",
    "--config",str(OVERFIT_CONFIG),
    "--resume","latest",
], cwd=PROJECT_ROOT, check=True)

In [ ]:
#@title 12. Inspect retained Drive checkpoints
from pathlib import Path
checkpoints = sorted(DRIVE_CHECKPOINT_ROOT.glob("checkpoint-*"), key=lambda p:int(p.name.split('-')[-1]))
print("Retained checkpoints:")
for path in checkpoints: print(path.name)
assert len(checkpoints) <= 3

In [ ]:
#@title 13. Sync final adapter and lightweight reports to Drive
import shutil
local_run = PROJECT_ROOT/"outputs/training"/OVERFIT_RUN_ID
drive_run = DRIVE_ROOT/"outputs/training"/OVERFIT_RUN_ID
drive_run.mkdir(parents=True, exist_ok=True)
for name in ["resolved_config.yaml","run_manifest.json","dataset_manifest.json","environment.json","trainable_parameters.json","one_batch_smoke_test.json"]:
    source=local_run/name
    if source.is_file(): shutil.copy2(source, drive_run/name)
if (local_run/"final_adapter").is_dir():
    shutil.copytree(local_run/"final_adapter", drive_run/"final_adapter", dirs_exist_ok=True)
print("Synced to", drive_run)

## Resume after a disconnect

Run cells 1–8 again, then run cell 11. The command uses `resume: latest`, checks
both local and Drive checkpoint roots, copies the newest Drive checkpoint back
to `/content`, and resumes optimizer/scheduler/trainer state.

The best checkpoint is selected by minimum `eval_loss`; the final Trainer state
records `best_model_checkpoint` and `best_metric`.